In [1]:
import pandas as pd
import numpy as np
import time
import os
import warnings
import statsmodels.api as sm
from scipy import stats
from pytrends.request import TrendReq
warnings.filterwarnings("ignore")

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================
TRENDS_FILE = "google_trends_btc.csv"
BTC_FILE    = "btc_prices.csv"
FINAL_FILE  = "trends_btc_final.csv"
KEYWORDS    = ["Bitcoin", "crypto crash", "buy bitcoin", "bitcoin price"]

# ============================================================
# STEP 1: FETCH GOOGLE TRENDS
# ============================================================
print("=" * 60)
print("STEP 1: Fetching Google Trends data")
print("=" * 60)

def fetch_trends(keywords, start="2019-01-01", end="2023-12-31"):
    """
    Fetch weekly Google Trends data for multiple keywords.
    Splits into yearly chunks to get weekly granularity.
    """
    pytrends = TrendReq(hl="en-US", tz=0, timeout=(10, 30))
    all_dfs = []

    years = [
        ("2019-01-01", "2019-12-31"),
        ("2020-01-01", "2020-12-31"),
        ("2021-01-01", "2021-12-31"),
        ("2022-01-01", "2022-12-31"),
        ("2023-01-01", "2023-12-31"),
    ]

    for year_start, year_end in years:
        timeframe = f"{year_start} {year_end}"
        print(f"  Fetching {timeframe}...")

        for attempt in range(3):
            try:
                pytrends.build_payload(
                    keywords,
                    cat=0,
                    timeframe=timeframe,
                    geo="",      # worldwide
                    gprop=""     # web search
                )
                df = pytrends.interest_over_time()

                if df.empty:
                    print(f"    ⚠️ Empty response for {year_start[:4]}")
                    break

                df = df.drop(columns=["isPartial"], errors="ignore")
                all_dfs.append(df)
                print(f"    ✅ {year_start[:4]}: {len(df)} weeks")
                break

            except Exception as e:
                print(f"    ⚠️ Attempt {attempt+1} failed: {e}")
                time.sleep(10)
        else:
            print(f"    ❌ Failed for {year_start[:4]}")

        time.sleep(2)  # be polite to Google

    if not all_dfs:
        return pd.DataFrame()

    combined = pd.concat(all_dfs)
    combined = combined[~combined.index.duplicated(keep="first")]
    combined = combined.sort_index()
    combined.index.name = "date"
    combined = combined.reset_index()
    combined["date"] = pd.to_datetime(combined["date"])
    return combined

if os.path.exists(TRENDS_FILE):
    print(f"  📂 Found existing {TRENDS_FILE} — skipping fetch.")
    trends = pd.read_csv(TRENDS_FILE)
else:
    trends = fetch_trends(KEYWORDS)
    if not trends.empty:
        trends.to_csv(TRENDS_FILE, index=False)
        print(f"\n  ✅ Saved {len(trends)} weeks to {TRENDS_FILE}")
    else:
        print("  ❌ No data fetched — try manual export from trends.google.com")

trends["date"] = pd.to_datetime(trends["date"])
print(f"\n  Columns: {list(trends.columns)}")
print(f"  Date range: {trends['date'].min().date()} → {trends['date'].max().date()}")
print(f"\n--- Preview ---")
print(trends.head(5).to_string())

STEP 1: Fetching Google Trends data
  Fetching 2019-01-01 2019-12-31...
    ✅ 2019: 53 weeks
  Fetching 2020-01-01 2020-12-31...
    ✅ 2020: 53 weeks
  Fetching 2021-01-01 2021-12-31...
    ✅ 2021: 53 weeks
  Fetching 2022-01-01 2022-12-31...
    ✅ 2022: 53 weeks
  Fetching 2023-01-01 2023-12-31...
    ✅ 2023: 53 weeks

  ✅ Saved 262 weeks to google_trends_btc.csv

  Columns: ['date', 'Bitcoin', 'crypto crash', 'buy bitcoin', 'bitcoin price']
  Date range: 2018-12-30 → 2023-12-31

--- Preview ---
        date  Bitcoin  crypto crash  buy bitcoin  bitcoin price
0 2018-12-30       35             0            1              8
1 2019-01-06       34             0            1              8
2 2019-01-13       33             0            1              7
3 2019-01-20       29             0            1              6
4 2019-01-27       29             0            1              7


In [3]:
# ============================================================
# STEP 2: LOAD BTC PRICES + RESAMPLE TO WEEKLY
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: Loading BTC prices & resampling to weekly")
print("=" * 60)

btc = pd.read_csv(BTC_FILE)
btc["date"] = pd.to_datetime(btc["date"])

# Resample to weekly (Google Trends is weekly)
btc_weekly = btc.set_index("date").resample("W-MON").agg({
    "close":      "last",
    "volume":     "sum",
    "return":     "sum",       # weekly return = sum of daily returns (approx)
    "volatility": "mean",      # avg daily volatility in the week
}).reset_index()

btc_weekly["weekly_return"]     = btc_weekly["close"].pct_change()
btc_weekly["weekly_volatility"] = btc_weekly["return"].rolling(4).std()

print(f"  ✅ Weekly BTC: {len(btc_weekly)} weeks")




STEP 2: Loading BTC prices & resampling to weekly
  ✅ Weekly BTC: 261 weeks


In [5]:
# ============================================================
# STEP 3: MERGE + FEATURE ENGINEERING
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: Merging & feature engineering")
print("=" * 60)

# Fix datetime precision before merge
btc_weekly["date"] = btc_weekly["date"].astype("datetime64[s]")
trends["date"] = trends["date"].astype("datetime64[s]")

df = pd.merge_asof(
    btc_weekly.sort_values("date"),
    trends.sort_values("date"),
    on="date",
    tolerance=pd.Timedelta("7d"),
    direction="nearest"
)

# Normalize trends columns to 0-100 range (already 0-100 from Google)
keyword_cols = [c for c in trends.columns if c != "date"]
print(f"  Trend keywords: {keyword_cols}")

# Composite attention index = mean of all keyword scores
df["trends_composite"] = df[keyword_cols].mean(axis=1)

# Lags
df["trends_lag1"] = df["trends_composite"].shift(1)
df["trends_lag4"] = df["trends_composite"].shift(4)  # 4 weeks lag
df["bitcoin_lag1"] = df["Bitcoin"].shift(1) if "Bitcoin" in df.columns else df[keyword_cols[0]].shift(1)

# Log transform (search interest is skewed)
df["trends_log"]      = np.log1p(df["trends_composite"])
df["trends_log_lag1"] = df["trends_log"].shift(1)

df.to_csv(FINAL_FILE, index=False)
print(f"  ✅ Final dataset: {len(df)} weeks → saved to {FINAL_FILE}")




STEP 3: Merging & feature engineering
  Trend keywords: ['Bitcoin', 'crypto crash', 'buy bitcoin', 'bitcoin price']
  ✅ Final dataset: 261 weeks → saved to trends_btc_final.csv


In [6]:
# ============================================================
# STEP 4: CORRELATION ANALYSIS
# ============================================================
print("\n" + "=" * 60)
print("STEP 4: Correlation Analysis")
print("=" * 60)

x_vars = {
    "trends_composite (same week)": "trends_composite",
    "trends_composite (lag 1w)":    "trends_lag1",
    "trends_composite (lag 4w)":    "trends_lag4",
    "trends_log (same week)":       "trends_log",
    "trends_log (lag 1w)":          "trends_log_lag1",
}
if "Bitcoin" in df.columns:
    x_vars["Bitcoin search (same week)"] = "Bitcoin"

for target_name, target_col in [("WEEKLY RETURN", "weekly_return"), ("WEEKLY VOLATILITY", "weekly_volatility")]:
    df_t = df.dropna(subset=[target_col, "trends_lag1"]).copy()
    print(f"\n  --- vs BTC {target_name} (n={len(df_t)}) ---")
    print(f"  {'Variable':<35} {'Pearson r':>10} {'p-val':>8} {'Spearman r':>11} {'p-val':>8}")
    print("  " + "=" * 68)
    for label, col in x_vars.items():
        if col not in df_t.columns:
            continue
        pr, pp = stats.pearsonr(df_t[col].fillna(0), df_t[target_col])
        sr, sp = stats.spearmanr(df_t[col].fillna(0), df_t[target_col])
        sig_p = "***" if pp < 0.01 else ("**" if pp < 0.05 else ("*" if pp < 0.1 else ""))
        sig_s = "***" if sp < 0.01 else ("**" if sp < 0.05 else ("*" if sp < 0.1 else ""))
        print(f"  {label:<35} {pr:>+10.4f} {pp:>7.4f}{sig_p:<3} {sr:>+10.4f} {sp:>7.4f}{sig_s:<3}")

print("\n  Significance: * p<0.1  ** p<0.05  *** p<0.01")




STEP 4: Correlation Analysis

  --- vs BTC WEEKLY RETURN (n=260) ---
  Variable                             Pearson r    p-val  Spearman r    p-val
  trends_composite (same week)           +0.0707  0.2559       +0.0537  0.3883   
  trends_composite (lag 1w)              +0.0339  0.5865       +0.0082  0.8952   
  trends_composite (lag 4w)              -0.0192  0.7582       -0.0657  0.2909   
  trends_log (same week)                 +0.0644  0.3012       +0.0537  0.3883   
  trends_log (lag 1w)                    +0.0292  0.6397       +0.0082  0.8952   
  Bitcoin search (same week)             +0.0823  0.1856       +0.0601  0.3348   

  --- vs BTC WEEKLY VOLATILITY (n=258) ---
  Variable                             Pearson r    p-val  Spearman r    p-val
  trends_composite (same week)           +0.1238  0.0469**     +0.1357  0.0294** 
  trends_composite (lag 1w)              +0.1689  0.0065***    +0.2102  0.0007***
  trends_composite (lag 4w)              +0.0450  0.4716       +0.0720  

In [7]:
# ============================================================
# STEP 5: OLS REGRESSION
# ============================================================
print("\n" + "=" * 60)
print("STEP 5: OLS Regression")
print("=" * 60)

df_ols = df.dropna(subset=[
    "weekly_return", "weekly_volatility",
    "trends_log_lag1", "trends_lag1", "bitcoin_lag1"
]).copy()
df_ols["vol_lag1"]    = df_ols["weekly_volatility"].shift(1)
df_ols["ret_lag1"]    = df_ols["weekly_return"].shift(1)
df_ols = df_ols.dropna(subset=["vol_lag1", "ret_lag1"])
print(f"  Observations: {len(df_ols)}")

y_vol = df_ols["weekly_volatility"]
y_ret = df_ols["weekly_return"]

# Model 1: Simple
X1 = sm.add_constant(df_ols[["trends_log_lag1"]])
model1 = sm.OLS(y_vol, X1).fit(cov_type="HC3")

# Model 2: Full multivariate with AR(1)
X2 = sm.add_constant(df_ols[[
    "trends_log_lag1", "trends_lag4",
    "vol_lag1", "ret_lag1"
]])
model2 = sm.OLS(y_vol, X2).fit(cov_type="HC3")

# Model 3: Without AR(1) — direct effect
X3 = sm.add_constant(df_ols[[
    "trends_log_lag1", "trends_lag4", "ret_lag1"
]])
model3 = sm.OLS(y_vol, X3).fit(cov_type="HC3")

# Model 4: Return prediction
X4 = sm.add_constant(df_ols[[
    "trends_log_lag1", "trends_lag4", "ret_lag1"
]])
model4 = sm.OLS(y_ret, X4).fit(cov_type="HC3")

# Summary Table
print("\n" + "=" * 70)
print("SUMMARY TABLE")
print("=" * 70)
print(f"  {'':28} {'Model 1':>10} {'Model 2':>10} {'Model 3':>10} {'Model 4':>10}")
print(f"  {'Dependent':28} {'Volat.':>10} {'Volat.':>10} {'Volat.':>10} {'Return':>10}")
print("  " + "-" * 65)

for v in ["trends_log_lag1", "trends_lag4", "vol_lag1", "ret_lag1", "const"]:
    row = f"  {v:28}"
    for model in [model1, model2, model3, model4]:
        if v in model.params:
            coef = model.params[v]
            pval = model.pvalues[v]
            sig  = "***" if pval < 0.01 else ("**" if pval < 0.05 else ("*" if pval < 0.1 else ""))
            row += f" {coef:>+8.4f}{sig:<2}"
        else:
            row += f" {'—':>10}"
    print(row)

print("  " + "-" * 65)
print(f"  {'R-squared':28} {model1.rsquared:>10.4f} {model2.rsquared:>10.4f} {model3.rsquared:>10.4f} {model4.rsquared:>10.4f}")
print(f"  {'Adj. R-squared':28} {model1.rsquared_adj:>10.4f} {model2.rsquared_adj:>10.4f} {model3.rsquared_adj:>10.4f} {model4.rsquared_adj:>10.4f}")
print(f"  {'Observations':28} {int(model1.nobs):>10} {int(model2.nobs):>10} {int(model3.nobs):>10} {int(model4.nobs):>10}")
print("  Significance: * p<0.1  ** p<0.05  *** p<0.01")
print("  Note: Robust standard errors (HC3)")
print("\n🎉 Google Trends pipeline complete!")


STEP 5: OLS Regression
  Observations: 257

SUMMARY TABLE
                                  Model 1    Model 2    Model 3    Model 4
  Dependent                        Volat.     Volat.     Volat.     Return
  -----------------------------------------------------------------
  trends_log_lag1               +0.0238***  +0.0219***  +0.0311***  +0.0187  
  trends_lag4                           —  -0.0011*   -0.0008    -0.0013  
  vol_lag1                              —  +0.7377***          —          —
  ret_lag1                              —  -0.0159    +0.0029    +0.0001  
  const                         +0.0165    -0.0211    +0.0083    -0.0169  
  -----------------------------------------------------------------
  R-squared                        0.0354     0.5712     0.0424     0.0047
  Adj. R-squared                   0.0316     0.5644     0.0310    -0.0072
  Observations                        257        257        257        257
  Significance: * p<0.1  ** p<0.05  *** p<0.01
  No